In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.multitest import multipletests

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COVARIATES, OUTCOMES

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('associations/effect_modification')
DISEASES = OUTCOMES
MODIFIERS = {'Deprivation (IMD)': 'z_imd_score', 'Air Pollution (NOx)': 'z_nox', 'Green Space (NDVI)': 'z_ndvi'}

data = load_data()

In [ ]:
# Fit outcome by modifier interactions
interaction_rows = []
interaction_models = {}
for outcome in DISEASES:
    for name, modifier in MODIFIERS.items():
        covariates_without_modifier = [covariate for covariate in COVARIATES if covariate != modifier]
        model_data = data.loc[complete_case_mask(data, outcome, [modifier, *covariates_without_modifier])]
        
        formula = f'{outcome} ~ walkability_score * {modifier} + ' + ' + '.join(covariates_without_modifier)
        model = smf.ols(formula, data = model_data).fit()
        interaction_term = f'walkability_score:{modifier}'
        ci = model.conf_int().loc[interaction_term]
        
        interaction_models[(outcome, name)] = (model, model_data)
        
        interaction_rows.append(
            {
                'outcome': outcome,
                'modifier': name,
                'interaction_term': interaction_term,
                'coefficient': model.params[interaction_term],
                'ci_lower': ci.iloc[0],
                'ci_upper': ci.iloc[1],
                'raw_p_value': model.pvalues[interaction_term]
            }
        )

interaction_results = pd.DataFrame(interaction_rows)

# Control FDR jointly
rejected, q_values, _, _ = multipletests(interaction_results['raw_p_value'], alpha = 0.05, method = 'fdr_bh')
interaction_results['fdr_q_value'] = q_values
interaction_results['significant'] = rejected

interaction_results.to_csv(OUT.interaction_models_csv, index = False)

display(interaction_results.round(4))

In [ ]:
# Plot adjusted prevalence by walkability and modifier level
min_w = data['walkability_score'].quantile(0.01)
max_w = data['walkability_score'].quantile(0.99)
w_range = np.linspace(min_w, max_w, 100)

colors = {'Low (-1 SD)': 'green', 'Medium (0 SD)': 'blue', 'High (+1 SD)': 'red'}
levels = {'Low (-1 SD)': -1, 'Medium (0 SD)': 0, 'High (+1 SD)': 1}

for outcome in DISEASES:
    fig, axes = plt.subplots(1, len(MODIFIERS), figsize = (18, 5), constrained_layout = True)

    for ax, (name, modifier) in zip(axes, MODIFIERS.items()):
        model, model_data = interaction_models[(outcome, name)]
        base_grid = pd.DataFrame({'walkability_score': w_range})

        for covariate in COVARIATES:
            base_grid[covariate] = model_data[covariate].mean()

        for label, level in levels.items():
            prediction_grid = base_grid.copy()
            prediction_grid[modifier] = level
            predictions = model.get_prediction(prediction_grid).summary_frame()
            
            ax.plot(w_range, predictions['mean'], color = colors[label], label = label)
            ax.fill_between(
                w_range, predictions['mean_ci_lower'], predictions['mean_ci_upper'], color = colors[label], alpha = 0.1
            )

        ax.set(
            title = f'{outcome.removeprefix("qof_")}: {name}', xlabel = 'Walkability score', ylabel = 'Predicted prevalence'
        )
        ax.legend()

    fig.savefig(OUT.figures / f'interactions_{outcome.removeprefix("qof_")}.png', dpi = 300, bbox_inches = 'tight')
    
    plt.show()

In [ ]:
# Interaction predictions
prediction_rows = []
for (outcome, modifier_label), (model, sample) in interaction_models.items():
    modifier = MODIFIERS[modifier_label]

    for level in [-1.0, 0.0, 1.0]:
        frame = pd.DataFrame({'walkability_score': w_range})

        for covariate in COVARIATES:
            frame[covariate] = sample[covariate].mean()

        frame[modifier] = level
        pred = model.get_prediction(frame).summary_frame()
        
        prediction_rows.append(
            pd.DataFrame(
                {
                    'outcome': outcome,
                    'modifier': modifier_label,
                    'level': level,
                    'walkability_score': w_range,
                    'predicted_prevalence': pred['mean'],
                    'ci_lower': pred.mean_ci_lower,
                    'ci_upper': pred.mean_ci_upper
                }
            )
        )

pd.concat(prediction_rows).to_csv(OUT.interaction_predictions_csv, index = False)